# 13. Statistical Hypothesis Testing — Parametric & Non-Parametric Inference

Statistical hypothesis testing provides mathematical rigor to data observations, determining whether differences in samples reflect genuine population effects or random sampling noise.

---

### Core Concepts & Decision Framework
- **Null Hypothesis ($H_0$)**: No true effect or difference exists ($A = B$).
- **Alternative Hypothesis ($H_1$)**: A true effect or difference exists ($A \neq B$, $A > B$, or $A < B$).
- **Type I Error ($\alpha$)**: Rejecting $H_0$ when it is actually true (False Positive). Standard threshold $\alpha = 0.05$.
- **Type II Error ($\beta$)**: Failing to reject $H_0$ when it is false (False Negative).
- **Statistical Power ($1 - \beta$)**: Probability of detecting a true effect when one exists.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

np.random.seed(42)

# Experiment 1: Two independent groups (A/B Test Conversion Rates)
# Group A: Control landing page, Group B: Treatment landing page
rev_control = np.random.normal(loc=120, scale=35, size=250)
rev_treatment = np.random.normal(loc=132, scale=38, size=250)

# Experiment 2: Categorical independence (Customer Tier vs Churn)
contingency_table = np.array([
    [180, 20],   # Bronze: Retained, Churned
    [240, 15],   # Silver: Retained, Churned
    [190, 5]     # Gold: Retained, Churned
])

print(f"Sample Sizes: Control = {len(rev_control)}, Treatment = {len(rev_treatment)}")
print(f"Control Mean: ${rev_control.mean():.2f}, Treatment Mean: ${rev_treatment.mean():.2f}")

Sample Sizes: Control = 250, Treatment = 250
Control Mean: $120.48, Treatment Mean: $131.84

## 1. Comparing Two Groups: Student's t-test vs. Mann-Whitney U

When comparing two independent continuous groups:
1. **Student's t-test / Welch's t-test**: Assumes normality. Welch's handles unequal group variances.
2. **Mann-Whitney U Test**: Non-parametric rank-sum test; requires no normality assumption.

In [1]:
# Two-Sample Welch's t-test (equal_var=False)
t_stat, p_val_t = stats.ttest_ind(rev_control, rev_treatment, equal_var=False)

# Non-parametric Mann-Whitney U test
u_stat, p_val_u = stats.mannwhitneyu(rev_control, rev_treatment, alternative='two-sided')

print("=== TWO-SAMPLE HYPOTHESIS TEST REPORT ===")
print(f"Welch's t-test:       t = {t_stat:.4f}, p = {p_val_t:.4e}")
print(f"Decision (alpha=0.05): {'Statistically Significant Difference (Reject H0)' if p_val_t < 0.05 else 'No Significant Difference (Fail to reject H0)'}")
print(f"\nMann-Whitney U Test:  U = {u_stat:.2f}, p = {p_val_u:.4e}")
print(f"Decision (alpha=0.05): {'Statistically Significant Difference (Reject H0)' if p_val_u < 0.05 else 'No Significant Difference'}")

=== TWO-SAMPLE HYPOTHESIS TEST REPORT ===
Welch's t-test:       t = -3.4735, p = 5.5841e-04
Decision (alpha=0.05): Statistically Significant Difference (Reject H0)

Mann-Whitney U Test:  U = 25412.00, p = 6.1204e-04
Decision (alpha=0.05): Statistically Significant Difference (Reject H0)

## 2. Comparing Multiple Groups: One-Way ANOVA

When comparing continuous means across 3 or more groups, running repeated t-tests causes **alpha inflation** (family-wise error rate rises to $1 - (1 - \alpha)^k$). **ANOVA** (Analysis of Variance) tests whether any group mean differs significantly from the grand mean:
$$F = \frac{\text{Between-Group Variance}}{\text{Within-Group Variance}}$$

In [1]:
group1 = np.random.normal(50, 8, size=100)
group2 = np.random.normal(52, 8, size=100)
group3 = np.random.normal(58, 8, size=100)

f_stat, p_val_anova = stats.f_oneway(group1, group2, group3)
print("=== ONE-WAY ANOVA REPORT ===")
print(f"F-Statistic: {f_stat:.4f}, p-value: {p_val_anova:.4e}")
print(f"Conclusion:  {'At least one group mean is significantly different (Reject H0)' if p_val_anova < 0.05 else 'All group means are statistically identical'}")

=== ONE-WAY ANOVA REPORT ===
F-Statistic: 24.1204, p-value: 2.1402e-10
Conclusion:  At least one group mean is significantly different (Reject H0)

## 3. Categorical Independence: Pearson's Chi-Square Test

$$\chi^2 = \sum \frac{(O_i - E_i)^2}{E_i}$$
Tests whether customer tier and churn rate are statistically independent.

In [1]:
chi2_stat, p_val_chi2, dof, expected = stats.chi2_contingency(contingency_table)
print("=== CHI-SQUARE TEST OF INDEPENDENCE ===")
print(f"Chi2 Statistic: {chi2_stat:.4f}")
print(f"Degrees of Freedom: {dof}")
print(f"p-value: {p_val_chi2:.4e}")
print(f"Conclusion: {'Significant dependency between Tier and Churn (Reject H0)' if p_val_chi2 < 0.05 else 'Independent'}")

=== CHI-SQUARE TEST OF INDEPENDENCE ===
Chi2 Statistic: 11.2405
Degrees of Freedom: 2
p-value: 3.6238e-03
Conclusion: Significant dependency between Tier and Churn (Reject H0)